# RAID v4b: Dense Sampling of Semantic Coherence Curves

Same metric as v4 (expected embedding similarity, shuffled-corrected) but with **dense window sampling** — 24 context windows from 2 to 128 tokens, concentrated in the 2-32 range where marginal gains are largest. Goal: find inflection points, slope changes, or transition regimes hidden by the sparse sampling in v4.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import json, math, time, gc, os, torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Imports OK")

In [ ]:
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATA = Path("/content/drive/MyDrive/LRTIA/Data/raid_sampled")
    DRIVE_RESULTS = Path("/content/drive/MyDrive/LRTIA/Results/RAID_v4b")
    if (DRIVE_DATA / "raid_corpus.jsonl").exists():
        DATA_DIR = DRIVE_DATA
    else:
        LOCAL_DATA = Path("/content/data/raid_sampled")
        if not (LOCAL_DATA / "raid_corpus.jsonl").exists():
            LOCAL_DATA.mkdir(parents=True, exist_ok=True)
            from google.colab import files
            uploaded = files.upload()
            for fname in uploaded:
                with open(LOCAL_DATA / fname, 'wb') as f:
                    f.write(uploaded[fname])
        DATA_DIR = LOCAL_DATA
    BASE_DIR = DRIVE_RESULTS
    BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    BASE_DIR = Path("../results/raid_v4b")
    DATA_DIR = Path("../data/raid_sampled")
    BASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR: {DATA_DIR}")
print(f"BASE_DIR: {BASE_DIR}")

MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True

# Dense windows: every 2 tokens from 2-32, then sparser to 128
WINDOWS = list(range(2, 34, 2)) + [40, 48, 56, 64, 80, 96, 112, 128]
BURN_IN = 128
MAX_SCORE_TOKENS = 64
BUFFER = 10
MIN_TOKENS = BURN_IN + MAX_SCORE_TOKENS + BUFFER

N_RANDOM_CONTROLS = 30
RANDOM_SEED = 42

DOMAINS = ['abstracts', 'books', 'news', 'poetry', 'recipes', 'reddit', 'reviews', 'wiki']

print(f"Windows ({len(WINDOWS)}): {WINDOWS}")
print(f"Min tokens: {MIN_TOKENS}")

In [ ]:
corpus_path = DATA_DIR / "raid_corpus.jsonl"
corpus = []
with open(corpus_path) as f:
    for line in f:
        corpus.append(json.loads(line))
print(f"Loaded {len(corpus)} documents")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto")
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model.eval()

embedding_matrix = model.get_input_embeddings().weight.detach()
embedding_matrix_normed = F.normalize(embedding_matrix, dim=-1)
print(f"Embedding matrix: {embedding_matrix.shape}")
print("Model loaded")

In [ ]:
@torch.no_grad()
def compute_expected_sim(token_ids, target_start, target_end):
    """Compute expected embedding similarity on target region. Focused metric only."""
    if target_end > len(token_ids):
        target_end = len(token_ids)
    if target_start >= target_end - 1:
        return None

    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]

    sim_sum = 0.0
    count = 0

    for i in range(target_start, target_end - 1):
        target_token = token_ids[i + 1]
        token_logits = logits[i]

        # Expected embedding similarity (prob-weighted, top-200)
        top200 = torch.topk(token_logits, 200)
        top200_probs = torch.softmax(top200.values, dim=-1)
        top200_embs = embedding_matrix_normed[top200.indices]
        expected_emb = torch.mv(top200_embs.t(), top200_probs)
        expected_emb = F.normalize(expected_emb, dim=-1)
        actual_emb = embedding_matrix_normed[target_token]
        sim_sum += torch.dot(expected_emb, actual_emb).item()
        count += 1

    del outputs, logits
    torch.cuda.empty_cache()

    return sim_sum / count if count > 0 else None


def compute_curve(token_ids):
    """Compute expected_sim at each window."""
    n_tokens = len(token_ids)
    if n_tokens <= BURN_IN:
        return None
    target_end = min(n_tokens, BURN_IN + MAX_SCORE_TOKENS)

    results = {}
    for W in WINDOWS:
        context_start = max(0, BURN_IN - W)
        actual_context = BURN_IN - context_start
        if actual_context < 2:
            continue
        truncated = token_ids[context_start:target_end]
        sim = compute_expected_sim(truncated, actual_context, len(truncated))
        if sim is not None:
            results[W] = sim

    return results if len(results) >= 3 else None


def extract_row(doc, curve, n_tokens):
    row = {
        'doc_id': doc['doc_id'],
        'domain': doc['domain'],
        'model': doc['model'],
        'population': doc['population'],
        'token_count': n_tokens,
    }
    for W, sim in sorted(curve.items()):
        row[f'esim_W{W}'] = sim
    return row


print("Functions defined")

In [ ]:
results_path = BASE_DIR / "essay_results_v4b.csv"

if results_path.exists():
    df = pd.read_csv(results_path)
    print(f"Loaded existing results: {len(df)} rows")
else:
    results = []
    skipped = 0
    for doc in tqdm(corpus, desc="Essays"):
        token_ids = tokenizer.encode(doc["text"], add_special_tokens=False)
        if len(token_ids) < MIN_TOKENS:
            skipped += 1
            continue
        curve = compute_curve(token_ids)
        if curve is None:
            skipped += 1
            continue
        results.append(extract_row(doc, curve, len(token_ids)))

    df = pd.DataFrame(results)
    df.to_csv(results_path, index=False)
    print(f"Processed {len(df)} essays ({skipped} skipped), saved to {results_path}")

print(f"Human: {len(df[df.population == 'human'])}, AI: {len(df[df.population == 'ai'])}")

In [ ]:
controls_path = BASE_DIR / "control_results_v4b.csv"

if controls_path.exists():
    df_ctrl = pd.read_csv(controls_path)
    print(f"Loaded existing controls: {len(df_ctrl)} rows")
else:
    rng = np.random.RandomState(RANDOM_SEED)
    human_docs = [d for d in corpus if d['population'] == 'human']
    sample_docs = rng.choice(human_docs, size=min(N_RANDOM_CONTROLS, len(human_docs)), replace=False)
    ctrl_rows = []

    rng_s = np.random.RandomState(RANDOM_SEED)
    for i, doc in enumerate(tqdm(sample_docs, desc="Shuffled")):
        token_ids = tokenizer.encode(doc["text"], add_special_tokens=False)
        if len(token_ids) < MIN_TOKENS:
            continue
        shuffled = list(token_ids)
        rng_s.shuffle(shuffled)
        curve = compute_curve(shuffled)
        if curve is None:
            continue
        ctrl_rows.append(extract_row(
            {'doc_id': f'shuffled_{i:03d}', 'domain': 'shuffled', 'model': 'shuffled', 'population': 'shuffled'},
            curve, len(token_ids)))

    rng_u = np.random.RandomState(RANDOM_SEED + 1)
    vocab_size = tokenizer.vocab_size
    rng_len = np.random.RandomState(RANDOM_SEED)
    sample_docs2 = rng_len.choice(human_docs, size=min(N_RANDOM_CONTROLS, len(human_docs)), replace=False)
    for i, doc in enumerate(tqdm(sample_docs2, desc="Uniform")):
        token_ids = tokenizer.encode(doc["text"], add_special_tokens=False)
        if len(token_ids) < MIN_TOKENS:
            continue
        uniform_ids = rng_u.randint(0, vocab_size, size=len(token_ids)).tolist()
        curve = compute_curve(uniform_ids)
        if curve is None:
            continue
        ctrl_rows.append(extract_row(
            {'doc_id': f'uniform_{i:03d}', 'domain': 'uniform', 'model': 'uniform', 'population': 'uniform'},
            curve, len(token_ids)))

    df_ctrl = pd.DataFrame(ctrl_rows)
    df_ctrl.to_csv(controls_path, index=False)
    print(f"Controls: {len(df_ctrl)} rows, saved to {controls_path}")

print(f"Shuffled: {len(df_ctrl[df_ctrl.population == 'shuffled'])}, Uniform: {len(df_ctrl[df_ctrl.population == 'uniform'])}")

In [ ]:
human = df[df.population == 'human']
ai = df[df.population == 'ai']
shuf = df_ctrl[df_ctrl.population == 'shuffled']
uni = df_ctrl[df_ctrl.population == 'uniform']

def get_means(sub):
    return np.array([sub[f'esim_W{w}'].mean() for w in WINDOWS if f'esim_W{w}' in sub.columns])

def get_corrected(sub, shuf_sub):
    return get_means(sub) - get_means(shuf_sub)

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# A: Raw curves, all conditions
ax = axes[0, 0]
for sub, label, color, ls, marker in [
    (human, 'Human', '#3498db', '-', 'o'),
    (ai, 'AI', '#e74c3c', '--', 'o'),
    (shuf, 'Shuffled', '#2ecc71', ':', 's'),
    (uni, 'Uniform', '#9b59b6', ':', 'D'),
]:
    vals = get_means(sub)
    ax.plot(WINDOWS[:len(vals)], vals, f'{marker}{ls}', color=color, linewidth=2, markersize=3, label=label)
ax.set_xscale('log', base=2)
ax.set_xlabel('Context Window (tokens)')
ax.set_ylabel('Expected Embedding Similarity')
ax.set_title('A. Raw Semantic Precision (dense sampling)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)

# B: Corrected curves
ax = axes[0, 1]
h_corr = get_corrected(human, shuf)
a_corr = get_corrected(ai, shuf)
ax.plot(WINDOWS[:len(h_corr)], h_corr, 'o-', color='#3498db', linewidth=2, markersize=3, label='Human')
ax.plot(WINDOWS[:len(a_corr)], a_corr, 'o--', color='#e74c3c', linewidth=2, markersize=3, label='AI')
ax.set_xscale('log', base=2)
ax.set_xlabel('Context Window (tokens)')
ax.set_ylabel('Corrected Semantic Precision')
ax.set_title('B. Coherence Signal (dense)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)

# C: Marginal gain per token (corrected)
ax = axes[1, 0]
for corr, label, color, ls in [(h_corr, 'Human', '#3498db', '-'), (a_corr, 'AI', '#e74c3c', '--')]:
    marg = []
    marg_x = []
    for i in range(1, len(WINDOWS)):
        dw = WINDOWS[i] - WINDOWS[i-1]
        dv = corr[i] - corr[i-1]
        marg.append(dv / dw)
        marg_x.append((WINDOWS[i] + WINDOWS[i-1]) / 2)
    ax.plot(marg_x, marg, f'o{ls}', color=color, linewidth=1.5, markersize=3, label=label)
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xscale('log', base=2)
ax.set_xlabel('Context Distance (tokens)')
ax.set_ylabel('Marginal Gain per Token')
ax.set_title('C. Influence Decay (dense)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)

# D: Normalized corrected, by genre
ax = axes[1, 1]
colors = plt.cm.Set2.colors
for i, d in enumerate(DOMAINS):
    sub = human[human.domain == d]
    corr = get_corrected(sub, shuf)
    total = corr[-1] - corr[0]
    if total > 0.001:
        norm = (corr - corr[0]) / total
        ax.plot(WINDOWS[:len(norm)], norm, 'o-', color=colors[i], linewidth=1.5, markersize=2, label=d)
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.4)
ax.plot([WINDOWS[0], WINDOWS[-1]], [0, 1], 'k--', alpha=0.2)
ax.set_xscale('log', base=2)
ax.set_ylim(-0.1, 1.1)
ax.set_xlabel('Context Window (tokens)')
ax.set_ylabel('Fraction of Coherence Gain')
ax.set_title('D. Genre Curves (dense)', fontweight='bold')
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.2)

plt.suptitle('Dense Sampling: Semantic Coherence Memory Curves (24 windows, 2-128 tokens)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig1_dense.png', dpi=150, bbox_inches='tight')
plt.show()

# Print the corrected values
print('\nCorrected expected_sim (Human - Shuffled):')
for i, w in enumerate(WINDOWS):
    if i < len(h_corr):
        print(f'  W{w:<4}: {h_corr[i]:.5f}')